#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "unit_price"
}

#Reading From Bronze

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

df.display()

# Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Null handling and Empty values

In [0]:
def is_empty(col):
    return col.isNull() | (F.trim(col) == "")

condition_invalid_business_keys  = (
    is_empty(F.col("sls_cust_id")) |
    is_empty(F.col("sls_prd_key")) |
    is_empty(F.col("sls_ord_num")) 
)

df_valid = df.filter(~condition_invalid_business_keys)
df_invalid = df.filter(condition_invalid_business_keys)

print("Total rows:", df.count())
print("Invalid rows:", df_invalid.count())
print("Valid rows:", df_valid.count())

df = df_valid

## Data type casting

In [0]:
df = (
    df
    .withColumn("sls_order_dt", F.try_to_date("sls_order_dt", "yyyyMMdd"))
    .withColumn("sls_ship_dt", F.try_to_date("sls_ship_dt", "yyyyMMdd"))
    .withColumn("sls_due_dt", F.try_to_date("sls_due_dt", "yyyyMMdd"))
)


## Handle duplicates

In [0]:

duplicate_count = (
    df.groupBy("sls_ord_num", "sls_prd_key")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate order-product pairs:", duplicate_count)

if duplicate_count > 0:
    raise Exception("Duplicate products found!")

duplicate_full = (
    df.groupBy(
        "sls_ord_num",
        "sls_cust_id",
        "sls_prd_key",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt",
        "sls_sales",
        "sls_quantity",
        "sls_price"
    )
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicates in Silver:", duplicate_full)

##Date validation

In [0]:

invalid_date_condition = (
    F.col("sls_order_dt").isNull() |
    F.col("sls_ship_dt").isNull() |
    F.col("sls_due_dt").isNull()
)

invalid_date_logic = df.filter(
    (F.col("sls_ship_dt") < F.col("sls_order_dt")) |
    (F.col("sls_due_dt") < F.col("sls_ship_dt"))
)

print("Invalid date relationships:", invalid_date_logic.count())

df_invalid = df.filter(invalid_date_condition)
df_valid = df.filter(~invalid_date_condition)

invalid_count = df_invalid.count()
total_count = df.count()

print("Total rows:", total_count)
print("Invalid date rows:", invalid_count)
print("Valid rows:", total_count - invalid_count)

df = df_valid



##Sales/price/quantity validation

In [0]:
display(df)

invalid_numeric_condition = (
    (F.col("sls_sales") < 0) |
    (F.col("sls_quantity") < 0) |
    (F.col("sls_price") < 0)
)

df_invalid_numeric = df.filter(invalid_numeric_condition)
df_valid_numeric = df.filter(~invalid_numeric_condition)

invalid_numeric_count = df_invalid_numeric.count()
total_count = df.count()

print("Total rows:", total_count)
print("Invalid numeric rows:", invalid_numeric_count)
print("Valid rows:", total_count - invalid_numeric_count)

df = df_valid_numeric

invalid_sales_condition = (
    F.abs(F.col("sls_sales") - F.col("sls_quantity") * F.col("sls_price")) > 0.01
)

df_invalid_sales = df.filter(invalid_sales_condition)
invalid_sales_count = df_invalid_sales.count()

print("Invalid sales calculation rows:", invalid_sales_count)

df = df.filter(~invalid_sales_condition)




## Renamig the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

    


## Sanity checks before write

In [0]:
def sanity_check(df):
    row_count = df.count()

    duplicate_order_product = (
        df.groupBy("order_number", "product_key")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    null_critical_fields = (
        df.filter(
            F.col("order_number").isNull() |
            F.col("product_key").isNull() |
            F.col("customer_id").isNull() |
            F.col("order_date").isNull()
        )
        .count()
    )

    invalid_date_logic = (
        df.filter(
            (F.col("ship_date") < F.col("order_date")) |
            (F.col("due_date") < F.col("ship_date")) |
            (F.col("due_date") < F.col("order_date"))
        )
        .count()
    )

    invalid_numeric = (
        df.filter(
            (F.col("sales_amount") < 0) |
            (F.col("quantity") < 0) |
            (F.col("unit_price") < 0)
        )
        .count()
    )

    invalid_sales_calc = (
        df.filter(
            F.abs(F.col("sales_amount") - F.col("quantity") * F.col("unit_price")) > 0.01
        )
        .count()
    )  

    return {
        "row_count": row_count,
        "duplicate_order_product": duplicate_order_product,
        "null_critical_fields": null_critical_fields,
        "invalid_date_logic": invalid_date_logic,
        "invalid_numeric": invalid_numeric,
        "invalid_sales_calc": invalid_sales_calc
    }


results = sanity_check(df)
print("Before write:", results)


# Write Into Silver

In [0]:
(df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.crm_sales_details"))

df_silver = spark.table("workspace.silver.crm_sales_details")

results = sanity_check(df_silver)
print("After write:", results)

if results["duplicate_order_product"] > 0:
    raise Exception("Duplicate order-product pairs found!")

if results["null_critical_fields"] > 0:
    raise Exception("Null critical fields found!")

if results["invalid_date_logic"] > 0:
    raise Exception("Invalid date relationships found!")

if results["invalid_numeric"] > 0:
    raise Exception("Invalid numeric values found!")

if results["invalid_sales_calc"] > 0:
    print("WARNING: Sales mismatch found")